# Seção 3 (configuração do modelo) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`03_coercao_strict_e_lax`](03_coercao_strict_e_lax.ipynb) | no 2.13, strict rejeita tupla→lista **e** lista→tupla (o curso dizia que tupla→lista passava) |
| [`04_validando_valores_default`](04_validando_valores_default.ipynb) | texto em célula de código no original |
| [`09_tratando_enums_python`](09_tratando_enums_python.ipynb) | comentário do YAML dizia `BANANA`, o arquivo tinha `apple` |
| [`12_projeto_solucao_comentada`](12_projeto_solucao_comentada.ipynb) | `type_` declarado duas vezes na tentativa original (vale a última) |

## Novas

**1. `extra="allow"` deixa passar dado não validado — até o banco.**

In [1]:
from datetime import date
from pydantic import BaseModel, ConfigDict, ValidationError


class Profile(BaseModel):
    model_config = ConfigDict(extra="allow")
    name: str


profile = Profile.model_validate({"name": "Ada", "is_admin": "true", "credit_limit": "1e9"})
print(f"profile.model_extra: {profile.model_extra}")
print(f"profile.model_dump(): {profile.model_dump()}")

profile.model_extra: {'is_admin': 'true', 'credit_limit': '1e9'}
profile.model_dump(): {'name': 'Ada', 'is_admin': 'true', 'credit_limit': '1e9'}


Os extras não são convertidos nem validados (`"true"` continua string) e **saem** no `model_dump()`. Se esse dump vira um `UPDATE`,
o cliente escreve em colunas que o modelo nem declara (*mass assignment*). `allow` só para dados que você repassa sem interpretar.

**2. Strict em Python ≠ strict em JSON.**

In [2]:
class Booking(BaseModel):
    model_config = ConfigDict(strict=True)
    day: date


try:
    Booking.model_validate({"day": "2026-01-01"})
except ValidationError as ex:
    print(f"strict, dict com string: {ex.errors()[0]['type']}")
booking_json = '{"day": "2026-01-01"}'
print(f"strict, JSON com string: {Booking.model_validate_json(booking_json)!r}")

strict, dict com string: date_type
strict, JSON com string: Booking(day=datetime.date(2026, 1, 1))


JSON não tem tipo data, então o strict de JSON aceita a string ISO. O mesmo payload passa por `model_validate_json` e falha em
`model_validate(json.loads(...))`. Valide JSON com `model_validate_json` — além de mais rápido, é o comportamento documentado.

**3. `str_max_length` vale para **todas** as strings — inclusive chaves de dicionário.**

In [3]:
class Labels(BaseModel):
    model_config = ConfigDict(str_max_length=5)
    code: str
    tags: dict[str, str]


try:
    Labels(code="ABC", tags={"prioridade": "alta"})
except ValidationError as ex:
    print(f"erro em: {ex.errors()[0]['loc']} ({ex.errors()[0]['type']})")

erro em: ('tags', 'prioridade', '[key]') (string_too_long)


Configurações `str_*` do modelo atingem chaves, itens de listas e campos aninhados. Restrições que valem para **um** campo vão no
`Field(max_length=...)` daquele campo.

**4. `frozen=True` não garante *hashable* nem imutabilidade profunda.**

In [4]:
class Team(BaseModel):
    model_config = ConfigDict(frozen=True)
    name: str
    members: list[str]


team = Team(name="core", members=["ana"])
team.members.append("bruno")
print(f"team.members depois de append: {team.members}")
try:
    {team}
except TypeError as ex:
    print(f"{{team}}: TypeError: {ex}")

team.members depois de append: ['ana', 'bruno']
{team}: TypeError: unhashable type: 'list'


`frozen` bloqueia atribuição (`team.name = ...`), não a mutação **dentro** dos campos, e o `hash` falha se algum campo for
*unhashable*. Para imutável de verdade: `tuple`/`frozenset` nos campos.

## Veja também

- [`dataclass_factory/04_politicas_e_presets`](../../../dataclass_factory/04_politicas_e_presets.ipynb) (caveats): `validate_assignment` não vê `append`,
  `str_to_lower` não vale para Enum, `model_copy(update=...)` pula validação, `use_enum_values` com default.